# Lab 4 : Implementation of Dynamic Programming

---


## Objective of the Lab

The objective of this lab is to understand, design, and implement various **Dynamic Programming (DP)** algorithms using Python. Through this lab, we aim to:

- Understand the working principle of Dynamic Programming (breaking a problem into overlapping subproblems, solving each subproblem once, and storing results to avoid recomputation).
- Implement DP solutions to classical graph, optimization, string, and scheduling problems.
- Analyze the time and space complexity of each implemented algorithm.
- Compare Dynamic Programming with greedy and brute-force approaches, and understand why DP is needed when greedy fails (e.g., 0/1 Knapsack) or brute force is too slow (e.g., TSP).


## Title of the Study

**Implementation and Analysis of Dynamic Programming Algorithms:**
All-Pairs Shortest Path (Floyd–Warshall), Travelling Salesman Problem, String Editing (Edit Distance), 0/1 Knapsack Problem, Matrix Chain Multiplication, and Flow Shop Scheduling.


## Related Theory (General)

**Dynamic Programming (DP)** is an algorithm design technique used to solve problems that exhibit:

1. **Overlapping subproblems** — the same subproblems are solved repeatedly in a naive recursive solution.
2. **Optimal substructure** — the optimal solution to the problem can be built from optimal solutions to its subproblems.

DP algorithms are typically implemented **top-down** (recursion + memoization) or **bottom-up** (iterative table filling), and trade extra memory (storing subproblem results) for a large reduction in time complexity compared to naive recursive/brute-force solutions.

In this lab, we study 6 classical DP algorithms:

1. All-Pairs Shortest Path — Floyd–Warshall Algorithm
2. Travelling Salesman Problem (Held–Karp DP)
3. String Editing — Edit Distance (Levenshtein Distance)
4. 0/1 Knapsack Problem
5. Matrix Chain Multiplication
6. Flow Shop Scheduling

Each algorithm is discussed below with its theory, diagram, source code, output, and complexity analysis.


---
## Q1. Program to Implement the All-Pairs Shortest Path Algorithm

### Related Theory
The **All-Pairs Shortest Path** problem asks for the shortest distance between **every pair** of vertices in a weighted graph (which may contain negative edge weights, but no negative cycles). The **Floyd–Warshall Algorithm** solves this using Dynamic Programming: it considers each vertex $k$ as a possible **intermediate** vertex on the path from $i$ to $j$, and updates:

$$dist[i][j] = \min(dist[i][j],\; dist[i][k] + dist[k][j])$$

for every pair $(i, j)$, iterating $k$ over all vertices. After considering all vertices as intermediates, `dist[i][j]` holds the true shortest distance.

### Related Diagram (Flow)
```
        Start
          |
   Read graph as adjacency (weight) matrix dist[][]
          |
   +-----------------------------------------------------------+
   | for k in vertices:                                        | <---+
   |   for i in vertices:                                      |     |
   |     for j in vertices:                                    |     |
   |       dist[i][j] = min(dist[i][j], dist[i][k]+dist[k][j]) | ----+
   +-----------------------------------------------------------+
          |
   Output dist[][] : shortest distance between every pair
          |
         End
```


In [1]:
def floyd_warshall(graph):
    """
    graph: dict {vertex: {neighbor: weight}} (adjacency dict).
    Returns: 2D dict dist[i][j] = shortest distance from i to j.
    """
    vertices = list(graph.keys())
    INF = float('inf')

    # Initialize distance table
    dist = {u: {v: INF for v in vertices} for u in vertices}
    for u in vertices:
        dist[u][u] = 0
        for v, w in graph[u].items():
            dist[u][v] = min(dist[u][v], w)

    # DP: try each vertex k as an intermediate point
    for k in vertices:
        for i in vertices:
            for j in vertices:
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]

    return dist


# Driver / Test
if __name__ == "__main__":
    graph = {
        'A': {'B': 3, 'C': 8, 'D': -4},
        'B': {'D': 1, 'A': float('inf')},
        'C': {'B': 4},
        'D': {'C': 2, 'A': float('inf')},
    }
    # fill missing edges with 'infinity' implicitly via dict access default
    for u in graph:
        for v in graph:
            if v not in graph[u]:
                graph[u][v] = float('inf')

    dist = floyd_warshall(graph)
    vertices = list(graph.keys())
    print("Shortest distance matrix (All-Pairs):")
    header = "      " + "".join(f"{v:>7}" for v in vertices)
    print(header)
    for u in vertices:
        row = f"{u:>4}: "
        for v in vertices:
            d = dist[u][v]
            row += f"{d:>7}" if d != float('inf') else f"{'inf':>7}"
        print(row)


Shortest distance matrix (All-Pairs):
            A      B      C      D
   A:       0      2     -2     -4
   B:     inf      0      3      1
   C:     inf      4      0      5
   D:     inf      6      2      0


### Analysis of the Algorithm
- **Time Complexity:** $O(V^3)$ — three nested loops over all vertices.
- **Space Complexity:** $O(V^2)$ — for the distance table.
- **Remark:** Floyd–Warshall works correctly with negative edge weights (unlike Dijkstra), as long as there is no negative-weight cycle. It is preferred over running Dijkstra/Bellman-Ford from every source when the graph is dense or when negative weights are present.


---
## Q2. Program to Implement the Travelling Salesman Problem

### Related Theory
The **Travelling Salesman Problem (TSP)** asks: given a set of cities and the distances between every pair, find the shortest possible route that visits every city exactly once and returns to the starting city. Brute force requires checking $(n-1)!$ permutations. The **Held–Karp Dynamic Programming algorithm** reduces this to $O(n^2 2^n)$ by using a **bitmask** to represent the subset of visited cities as the DP state:

$$dp[S][j] = \min_{i \in S,\, i \ne j} \big( dp[S \setminus \{j\}][i] + dist(i, j) \big)$$

where $S$ is the set of visited cities and $j$ is the last city visited. This is still exponential, but is dramatically faster than the factorial brute-force approach for moderate $n$.

### Related Diagram (Flow)
```
        Start
          |
   Read distance matrix dist[n][n]
          |
   dp[{start}][start] = 0
          |
   +--------------------------------------------------------------+
   | for each subset S of cities (containing start):              | <---+
   |   for each city j in S, j != start:                          |     |
   |     dp[S][j] = min over i in S,i!=j of                       |     |
   |                 dp[S-{j}][i] + dist[i][j]                    | ----+
   +--------------------------------------------------------------+
          |
   answer = min over j of dp[FullSet][j] + dist[j][start]
          |
   Output minimum tour cost
          |
         End
```


In [2]:
def tsp_held_karp(dist):
    """
    dist: 2D list, dist[i][j] = distance from city i to city j.
    Returns: (min_cost, optimal_path) for the TSP tour starting/ending at city 0.
    """
    n = len(dist)
    ALL_VISITED = (1 << n) - 1

    # dp[mask][j] = min cost to have visited exactly 'mask' cities, ending at city j
    dp = [[float('inf')] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]
    dp[1][0] = 0    # start at city 0, only city 0 visited

    for mask in range(1 << n):
        for j in range(n):
            if not (mask & (1 << j)) or dp[mask][j] == float('inf'):
                continue
            for k in range(n):
                if mask & (1 << k):
                    continue                 # k already visited
                new_mask = mask | (1 << k)
                new_cost = dp[mask][j] + dist[j][k]
                if new_cost < dp[new_mask][k]:
                    dp[new_mask][k] = new_cost
                    parent[new_mask][k] = j

    # Close the tour: return to city 0 from the last visited city
    best_cost, last = float('inf'), -1
    for j in range(1, n):
        cost = dp[ALL_VISITED][j] + dist[j][0]
        if cost < best_cost:
            best_cost, last = cost, j

    # Reconstruct path (walk backward through parent pointers to city 0)
    path, mask = [], ALL_VISITED
    while last != -1:
        path.append(last)
        prev = parent[mask][last]
        mask ^= (1 << last)
        last = prev
    path.reverse()
    path.append(0)   # return to the starting city to complete the tour

    return best_cost, path


# Driver / Test
if __name__ == "__main__":
    dist = [
        [0, 10, 15, 20],
        [10, 0, 35, 25],
        [15, 35, 0, 30],
        [20, 25, 30, 0],
    ]

    min_cost, path = tsp_held_karp(dist)
    print(f"Optimal tour: {' -> '.join(map(str, path))}")
    print(f"Minimum tour cost: {min_cost}")


Optimal tour: 0 -> 2 -> 3 -> 1 -> 0
Minimum tour cost: 80


### Analysis of the Algorithm
- **Time Complexity:** $O(n^2 \, 2^n)$ — for every subset ($2^n$) and every ending city ($n$), we try every transition city ($n$).
- **Space Complexity:** $O(n \, 2^n)$ — for the DP table indexed by (subset, last city).
- **Remark:** The Held–Karp DP is a dramatic improvement over the $O(n!)$ brute-force approach, but it is still exponential and only practical for a small number of cities (roughly $n \le 20$). TSP is **NP-hard**, so no known polynomial-time exact algorithm exists.


---
## Q3. Program to Implement String Editing (Edit Distance)

### Related Theory
The **Edit Distance** (Levenshtein Distance) between two strings is the minimum number of single-character **insertions**, **deletions**, or **substitutions** required to transform one string into the other. Using DP, we define `dp[i][j]` as the edit distance between the first `i` characters of string `A` and the first `j` characters of string `B`:

$$dp[i][j] = \begin{cases} dp[i-1][j-1] & \text{if } A[i]=B[j] \\ 1 + \min(dp[i-1][j],\, dp[i][j-1],\, dp[i-1][j-1]) & \text{otherwise} \end{cases}$$

### Related Diagram (Flow)
```
        Start
          |
   Read strings A (len m), B (len n)
          |
   dp[i][0] = i for all i;  dp[0][j] = j for all j
          |
   +--------------------------------------------------------------+
   | for i in 1..m:                                                | <---+
   |   for j in 1..n:                                              |     |
   |     if A[i]==B[j]: dp[i][j] = dp[i-1][j-1]                    |     |
   |     else: dp[i][j] = 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])| ----+
   +--------------------------------------------------------------+
          |
   Output dp[m][n]  (edit distance)
          |
         End
```


In [3]:
def edit_distance(A, B):
    """Return the minimum edit distance (Levenshtein distance) between strings A and B."""
    m, n = len(A), len(B)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i          # delete all i characters of A
    for j in range(n + 1):
        dp[0][j] = j          # insert all j characters of B

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if A[i - 1] == B[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],       # deletion
                    dp[i][j - 1],       # insertion
                    dp[i - 1][j - 1],   # substitution
                )
    return dp[m][n]


# Driver / Test
if __name__ == "__main__":
    pairs = [("kitten", "sitting"), ("sunday", "saturday"), ("intention", "execution")]
    for A, B in pairs:
        d = edit_distance(A, B)
        print(f'Edit distance between "{A}" and "{B}": {d}')


Edit distance between "kitten" and "sitting": 3
Edit distance between "sunday" and "saturday": 3
Edit distance between "intention" and "execution": 5


### Analysis of the Algorithm
- **Time Complexity:** $O(m \times n)$ — filling the entire DP table of size $(m+1) \times (n+1)$.
- **Space Complexity:** $O(m \times n)$ (can be optimized to $O(\min(m,n))$ by keeping only the previous row).
- **Remark:** Edit Distance is widely used in spell-checking, DNA sequence alignment, and diff tools; the DP table also allows reconstructing the actual sequence of edit operations, not just the count.


---
## Q4. Program to Implement 0/1 Knapsack Problem using Dynamic Programming

### Related Theory
In the **0/1 Knapsack Problem**, each item must be taken **entirely or not at all** (no fractions allowed). Given `n` items with weights $w_i$ and values $v_i$, and capacity $W$, we define:

$$dp[i][w] = \max(dp[i-1][w],\; dp[i-1][w-w_i] + v_i) \quad \text{if } w_i \le w$$
$$dp[i][w] = dp[i-1][w] \quad \text{otherwise}$$

Unlike the fractional version, greedy fails here because taking the best ratio item first can leave wasted capacity; DP guarantees the optimal solution by exploring the take/don't-take decision for every item.

### Related Diagram (Flow)
```
        Start
          |
   Read items (weight, value), capacity W
          |
   dp[0][w] = 0 for all w  (no items => 0 value)
          |
   +-----------------------------------------------------------------+
   | for i in 1..n:                                                  | <---+
   |   for w in 0..W:                                                |     |
   |     if weight[i] <= w:                                          |     |
   |       dp[i][w] = max(dp[i-1][w], dp[i-1][w-weight[i]]+value[i]) |     |
   |     else: dp[i][w] = dp[i-1][w]                                 | ----+
   +-----------------------------------------------------------------+
          |
   Output dp[n][W]  (maximum value)
          |
         End
```


In [4]:
def knapsack_01(items, capacity):
    """
    items: list of tuples (name, weight, value)
    capacity: knapsack capacity (integer)
    Returns: (max_value, selected_items)
    """
    n = len(items)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        name, weight, value = items[i - 1]
        for w in range(capacity + 1):
            if weight <= w:
                dp[i][w] = max(dp[i - 1][w], dp[i - 1][w - weight] + value)
            else:
                dp[i][w] = dp[i - 1][w]

    # Backtrack to find which items were selected
    selected = []
    w = capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i - 1][w]:      # item i was included
            name, weight, value = items[i - 1]
            selected.append(name)
            w -= weight
    selected.reverse()

    return dp[n][capacity], selected


# Driver / Test
if __name__ == "__main__":
    items = [("Item1", 2, 3), ("Item2", 3, 4), ("Item3", 4, 5), ("Item4", 5, 6)]
    capacity = 5

    max_value, selected = knapsack_01(items, capacity)
    print(f"Selected items: {selected}")
    print(f"Maximum value obtainable: {max_value}")


Selected items: ['Item1', 'Item2']
Maximum value obtainable: 7


### Analysis of the Algorithm
- **Time Complexity:** $O(n \times W)$ — filling the DP table of size $(n+1) \times (W+1)$.
- **Space Complexity:** $O(n \times W)$ (can be optimized to $O(W)$ using a 1-D rolling array).
- **Remark:** This is a **pseudo-polynomial** time algorithm — it is efficient in practice for reasonably sized $W$, but is technically exponential in the number of *bits* used to represent $W$. The 0/1 Knapsack Problem is NP-complete in general; DP works because $W$ is typically bounded/small in practice.


---
## Q5. Program to Implement Matrix Chain Multiplication

### Related Theory
Given a chain of matrices $A_1, A_2, \ldots, A_n$ with dimensions given by an array $p$ (matrix $A_i$ has dimensions $p_{i-1} \times p_i$), **Matrix Chain Multiplication** finds the parenthesization (order of multiplication) that minimizes the total number of scalar multiplications. Multiplying matrices is associative, so different orders give the same result but with very different costs.

We define $dp[i][j]$ as the minimum cost of multiplying matrices $A_i \ldots A_j$:

$$dp[i][j] = \min_{i \le k < j} \big( dp[i][k] + dp[k+1][j] + p_{i-1} \cdot p_k \cdot p_j \big)$$

### Related Diagram (Flow)
```
        Start
          |
   Read dimensions array p[0..n]
          |
   dp[i][i] = 0 for all i  (single matrix, no cost)
          |
   +----------------------------------------------------------------+
   | for length L = 2 to n:                                         | <---+
   |   for i = 1 to n-L+1:  j = i+L-1                               |     |
   |     dp[i][j] = min over k in [i, j-1] of                       |     |
   |        dp[i][k] + dp[k+1][j] + p[i-1]*p[k]*p[j]                | ----+
   +----------------------------------------------------------------+
          |
   Output dp[1][n]  (minimum scalar multiplications)
          |
         End
```


In [5]:
def matrix_chain_order(p):
    """
    p: list of matrix dimensions such that matrix i has dimensions p[i-1] x p[i]
       (so len(p) = number_of_matrices + 1)
    Returns: (min_cost, optimal_parenthesization_string)
    """
    n = len(p) - 1                       # number of matrices
    dp = [[0] * (n + 1) for _ in range(n + 1)]
    split = [[0] * (n + 1) for _ in range(n + 1)]

    for length in range(2, n + 1):       # chain length
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = float('inf')
            for k in range(i, j):
                cost = dp[i][k] + dp[k + 1][j] + p[i - 1] * p[k] * p[j]
                if cost < dp[i][j]:
                    dp[i][j] = cost
                    split[i][j] = k

    def build_parens(i, j):
        if i == j:
            return f"A{i}"
        k = split[i][j]
        return f"({build_parens(i, k)} x {build_parens(k + 1, j)})"

    return dp[1][n], build_parens(1, n)


# Driver / Test
if __name__ == "__main__":
    # Matrices: A1(40x20), A2(20x30), A3(30x10), A4(10x30)
    p = [40, 20, 30, 10, 30]

    min_cost, parenthesization = matrix_chain_order(p)
    print(f"Optimal parenthesization: {parenthesization}")
    print(f"Minimum number of scalar multiplications: {min_cost}")


Optimal parenthesization: ((A1 x (A2 x A3)) x A4)
Minimum number of scalar multiplications: 26000


### Analysis of the Algorithm
- **Time Complexity:** $O(n^3)$ — three nested loops (chain length, start index, split point).
- **Space Complexity:** $O(n^2)$ — for the `dp` and `split` tables.
- **Remark:** Matrix Chain Multiplication does not compute the product itself — it only determines the **optimal order** of multiplication to minimize scalar multiplication operations, which can yield dramatic savings for long chains of matrices with widely varying dimensions.


---
## Q6. Program to Implement Flow Shop Scheduling

### Related Theory
**Flow Shop Scheduling** deals with scheduling `n` jobs that must each pass through the **same sequence of machines** (e.g., Machine 1 then Machine 2), where every job follows the same machine order. For the classic **2-machine flow shop** problem, **Johnson's Rule** (a DP/greedy hybrid) gives the optimal schedule that minimizes the total completion time (makespan):

1. Find the job with the smallest processing time across both machines.
2. If that time is on Machine 1, schedule the job as early as possible; if on Machine 2, schedule it as late as possible.
3. Remove the job and repeat until all jobs are scheduled.

We then compute the resulting makespan using a DP-style forward pass that tracks the completion time of each job on each machine.

### Related Diagram (Flow)
```
        Start
          |
   Read jobs with (time on M1, time on M2)
          |
   +----------------------------------------------------------+
   | while unscheduled jobs remain:                           | <---+
   |   find job with min(time_M1, time_M2) among remaining    |     |
   |   if min is time_M1: place job at front of schedule      |     |
   |   else: place job at back of schedule                    |     |
   |   remove job from remaining                              | ----+
   +----------------------------------------------------------+
          |
   Compute completion times (DP forward pass) for M1, M2
          |
   Output schedule order and total makespan
          |
         End
```


In [6]:
def johnsons_rule(jobs):
    """
    jobs: list of tuples (job_id, time_m1, time_m2)
    Returns: (schedule_order, makespan)
    """
    remaining = jobs[:]
    front, back = [], []

    while remaining:
        # Find job with the overall minimum processing time (either machine)
        job = min(remaining, key=lambda j: min(j[1], j[2]))
        job_id, t1, t2 = job
        if t1 <= t2:
            front.append(job)          # schedule as early as possible
        else:
            back.insert(0, job)        # schedule as late as possible
        remaining.remove(job)

    schedule = front + back

    # Compute makespan using a DP-style forward pass over the two machines
    n = len(schedule)
    completion_m1 = [0] * n
    completion_m2 = [0] * n

    for i, (job_id, t1, t2) in enumerate(schedule):
        completion_m1[i] = t1 if i == 0 else completion_m1[i - 1] + t1
        if i == 0:
            completion_m2[i] = completion_m1[i] + t2
        else:
            completion_m2[i] = max(completion_m1[i], completion_m2[i - 1]) + t2

    makespan = completion_m2[-1]
    schedule_order = [job[0] for job in schedule]
    return schedule_order, makespan


# Driver / Test
if __name__ == "__main__":
    # jobs: (job_id, time on Machine 1, time on Machine 2)
    jobs = [("J1", 5, 2), ("J2", 1, 6), ("J3", 9, 7), ("J4", 3, 8), ("J5", 10, 4)]

    schedule_order, makespan = johnsons_rule(jobs)
    print(f"Optimal job sequence (Johnson's Rule): {schedule_order}")
    print(f"Total makespan (completion time): {makespan}")


Optimal job sequence (Johnson's Rule): ['J2', 'J4', 'J3', 'J5', 'J1']
Total makespan (completion time): 30


### Analysis of the Algorithm
- **Time Complexity:** $O(n^2)$ for the naive job-selection loop shown above (can be improved to $O(n \log n)$ by sorting jobs cleverly); the makespan computation pass is $O(n)$.
- **Space Complexity:** $O(n)$ for the schedule and completion-time arrays.
- **Remark:** Johnson's Rule gives the **provably optimal** schedule for the **2-machine** flow shop problem. Flow shop scheduling with **3 or more machines** in general is **NP-hard**, and heuristic or DP/branch-and-bound approaches are needed instead of a simple closed-form rule.


---
## Discussion and Conclusion

In this lab, we implemented and analyzed six classical Dynamic Programming algorithms:

- **Floyd–Warshall** computed all-pairs shortest paths in $O(V^3)$ using a simple triple-loop DP over intermediate vertices, correctly handling negative edge weights.
- **Travelling Salesman Problem (Held–Karp)** reduced the brute-force $O(n!)$ search to $O(n^2 2^n)$ using bitmask DP, though it remains exponential since TSP is NP-hard.
- **Edit Distance** demonstrated classic 2-D string DP, widely applicable to spell-checkers and sequence alignment.
- **0/1 Knapsack** showed why greedy fails for indivisible items, and how DP guarantees the optimal solution in pseudo-polynomial time $O(nW)$.
- **Matrix Chain Multiplication** illustrated interval DP, finding the optimal multiplication order in $O(n^3)$ instead of exploring all possible parenthesizations (which is exponential).
- **Flow Shop Scheduling** (Johnson's Rule) combined a greedy ordering insight with a DP-style forward pass to minimize makespan for the 2-machine case.

**Overall Conclusion:**

This lab reinforced how Dynamic Programming systematically avoids the redundant recomputation inherent in naive recursive/brute-force solutions, by identifying overlapping subproblems and optimal substructure. We observed a recurring theme: many problems that are exponential in their naive form (TSP, matrix chain order, knapsack) become tractable — polynomial or, at worst, dramatically reduced exponential — once an appropriate DP state and recurrence are identified. At the same time, problems like TSP and general Flow Shop Scheduling remind us that DP does not eliminate NP-hardness; it merely provides the best exact algorithm currently known for a given problem structure. This lab strengthened our ability to recognize DP-friendly problem structure and to design efficient recurrences and tables.
